In [1]:
#Step 1: Import Required Libraries
import pandas as pd
import json
import joblib
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import spacy



In [2]:
#Step 2: Load ESG Dataset
df=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ESG_daily_news.csv')
df=df.dropna(subset=['headline', 'text', 'Date']).reset_index(drop=True)
display(df)

,Date,headline,text
0,2022-11-28,Top-Ranked Hedge Fund Makes Contrarian Bet on ...,As most techology stocks reel from higher inte...
1,2022-11-27,Deutsche Bank’s DWS CEO Mulls New Legal Setup,DWS Group CEO Stefan Hoops is considering chan...
2,2022-11-24,"JPMorgan, Deutsche Bank Sued by Epstein Accusers",JPMorgan Chase & Co. and Deutsche Bank AG were...
3,2022-11-23,Tech Job Cuts Increase ‘Anxiety’ Across Industry,"After years of exuberant growth and hiring, la..."
4,2022-11-22,"Amundi, DWS Reclassify Funds in Major Industry...",Amundi and Deutsche Bank’s DWS Group are downg...
...,...,...,...
339,2021-10-28,Citi Pitches $1 Billion Social Bond Amid Race ...,Citigroup Inc. is returning to the social bond...
340,2021-10-26,Jet Fuel Surges in Price as Travel Restriction...,Jet fuel is back in a big way. The oil product...
341,2021-10-25,Rich Nations Fail to Meet Climate Target Befor...,Rich countries have failed to meet their pledg...
342,2021-10-24,Negotiators Edge Closer to Global Carbon Marke...,Nations are edging toward a deal that might cr...


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
#Install Dependancies
!pip install transformers
!pip install torch

In [5]:
#Preview Dataset
display(df)

,Date,headline,text
0,2022-11-28,Top-Ranked Hedge Fund Makes Contrarian Bet on ...,As most techology stocks reel from higher inte...
1,2022-11-27,Deutsche Bank’s DWS CEO Mulls New Legal Setup,DWS Group CEO Stefan Hoops is considering chan...
2,2022-11-24,"JPMorgan, Deutsche Bank Sued by Epstein Accusers",JPMorgan Chase & Co. and Deutsche Bank AG were...
3,2022-11-23,Tech Job Cuts Increase ‘Anxiety’ Across Industry,"After years of exuberant growth and hiring, la..."
4,2022-11-22,"Amundi, DWS Reclassify Funds in Major Industry...",Amundi and Deutsche Bank’s DWS Group are downg...
...,...,...,...
339,2021-10-28,Citi Pitches $1 Billion Social Bond Amid Race ...,Citigroup Inc. is returning to the social bond...
340,2021-10-26,Jet Fuel Surges in Price as Travel Restriction...,Jet fuel is back in a big way. The oil product...
341,2021-10-25,Rich Nations Fail to Meet Climate Target Befor...,Rich countries have failed to meet their pledg...
342,2021-10-24,Negotiators Edge Closer to Global Carbon Marke...,Nations are edging toward a deal that might cr...


In [6]:
#Set Up the NER Pipeline using finbert-ner
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("Jean-Baptiste/roberta-large-ner-english")
model = AutoModelForTokenClassification.from_pretrained("Jean-Baptiste/roberta-large-ner-english")

# Initialize NER pipeline
ner_pipe = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

# Company extraction function
def extract_companies(text):
    entities = ner_pipe(text)
    orgs = [ent["word"] for ent in entities if ent["entity_group"] == "ORG"]
    return orgs if orgs else ["unknown"]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cuda:0


In [7]:
#Apply to Headlines

from tqdm import tqdm
tqdm.pandas()

df["companies"] = df["headline"].progress_apply(extract_companies)
df["company"] = df["companies"].apply(lambda x: ", ".join(x))
df.to_csv("ESG_news_with_companies.csv", index=False)

  0%|          | 0/344 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
100%|██████████| 344/344 [00:16<00:00, 21.24it/s]


In [8]:
#Preview the dataset
display(df)

,Date,headline,text,companies,company
0,2022-11-28,Top-Ranked Hedge Fund Makes Contrarian Bet on ...,As most techology stocks reel from higher inte...,[unknown],unknown
1,2022-11-27,Deutsche Bank’s DWS CEO Mulls New Legal Setup,DWS Group CEO Stefan Hoops is considering chan...,"[ Deutsche Bank, DWS]","Deutsche Bank, DWS"
2,2022-11-24,"JPMorgan, Deutsche Bank Sued by Epstein Accusers",JPMorgan Chase & Co. and Deutsche Bank AG were...,"[ JPMorgan, Deutsche Bank]","JPMorgan, Deutsche Bank"
3,2022-11-23,Tech Job Cuts Increase ‘Anxiety’ Across Industry,"After years of exuberant growth and hiring, la...",[unknown],unknown
4,2022-11-22,"Amundi, DWS Reclassify Funds in Major Industry...",Amundi and Deutsche Bank’s DWS Group are downg...,"[ Amundi, DWS]","Amundi, DWS"
...,...,...,...,...,...
339,2021-10-28,Citi Pitches $1 Billion Social Bond Amid Race ...,Citigroup Inc. is returning to the social bond...,[ Citi],Citi
340,2021-10-26,Jet Fuel Surges in Price as Travel Restriction...,Jet fuel is back in a big way. The oil product...,[unknown],unknown
341,2021-10-25,Rich Nations Fail to Meet Climate Target Befor...,Rich countries have failed to meet their pledg...,[unknown],unknown
342,2021-10-24,Negotiators Edge Closer to Global Carbon Marke...,Nations are edging toward a deal that might cr...,[unknown],unknown


In [9]:
# Create a unified content field for embedding
df['content'] = (
    "Company: " + df['company'].fillna('') + "\n"
    "Date: " + df['Date'].astype(str) + "\n"
    "Headline: " + df['headline'].fillna('') + "\n"
    "Text: " + df['text'].fillna('')
)

In [10]:
display(df)

,Date,headline,text,companies,company,content
0,2022-11-28,Top-Ranked Hedge Fund Makes Contrarian Bet on ...,As most techology stocks reel from higher inte...,[unknown],unknown,Company: unknown\nDate: 2022-11-28\nHeadline: ...
1,2022-11-27,Deutsche Bank’s DWS CEO Mulls New Legal Setup,DWS Group CEO Stefan Hoops is considering chan...,"[ Deutsche Bank, DWS]","Deutsche Bank, DWS","Company: Deutsche Bank, DWS\nDate: 2022-11-2..."
2,2022-11-24,"JPMorgan, Deutsche Bank Sued by Epstein Accusers",JPMorgan Chase & Co. and Deutsche Bank AG were...,"[ JPMorgan, Deutsche Bank]","JPMorgan, Deutsche Bank","Company: JPMorgan, Deutsche Bank\nDate: 2022..."
3,2022-11-23,Tech Job Cuts Increase ‘Anxiety’ Across Industry,"After years of exuberant growth and hiring, la...",[unknown],unknown,Company: unknown\nDate: 2022-11-23\nHeadline: ...
4,2022-11-22,"Amundi, DWS Reclassify Funds in Major Industry...",Amundi and Deutsche Bank’s DWS Group are downg...,"[ Amundi, DWS]","Amundi, DWS","Company: Amundi, DWS\nDate: 2022-11-22\nHead..."
...,...,...,...,...,...,...
339,2021-10-28,Citi Pitches $1 Billion Social Bond Amid Race ...,Citigroup Inc. is returning to the social bond...,[ Citi],Citi,Company: Citi\nDate: 2021-10-28\nHeadline: Ci...
340,2021-10-26,Jet Fuel Surges in Price as Travel Restriction...,Jet fuel is back in a big way. The oil product...,[unknown],unknown,Company: unknown\nDate: 2021-10-26\nHeadline: ...
341,2021-10-25,Rich Nations Fail to Meet Climate Target Befor...,Rich countries have failed to meet their pledg...,[unknown],unknown,Company: unknown\nDate: 2021-10-25\nHeadline: ...
342,2021-10-24,Negotiators Edge Closer to Global Carbon Marke...,Nations are edging toward a deal that might cr...,[unknown],unknown,Company: unknown\nDate: 2021-10-24\nHeadline: ...


In [11]:
#Generate Embeddings using SentenceTransformer
# Initialize sentence embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

#Create embeddings from content

embeddings=model.encode(df['content'].tolist(), show_progress_bar=True)

#Save embeddings and metadata

joblib.dump(embeddings, 'esg_embeddings.pkl')
joblib.dump(df.to_dict(orient='records'), "esg_metadata.pkl")

print("✅ Embeddings and metadata saved.")

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


✅ Embeddings and metadata saved.


In [12]:
#Load Saved Embeddings & Metadata
import joblib
import pandas as pd

# Load embeddings and metadata
embeddings = joblib.load("esg_embeddings.pkl")
metadata = joblib.load("esg_metadata.pkl")
df = pd.DataFrame(metadata)

In [13]:
!pip install faiss-cpu --quiet

In [14]:
#step:02 Build Semantic Search Index using FAISS
import faiss
import numpy as np

# Create FAISS index
dimension = len(embeddings[0])
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(np.array(embeddings))

In [15]:
#Define Semantic Search Function
from sentence_transformers import SentenceTransformer
import numpy as np

# (Re)load embedding model if not in memory
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def semantic_search(query, df, faiss_index, embedding_model, top_k=3):
    # 1. Embed the query
    q_vec = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    # 2. Search FAISS index
    distances, indices = faiss_index.search(q_vec, top_k)
    # 3. Return the matching metadata rows
    return df.iloc[indices[0]]

In [16]:
#STEP-04:Set Up FLAN-T5 for Answer Generation
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load FLAN-T5 (base for lighter footprint)
flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
flan_model     = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generate_answer(context, question, max_new_tokens=128):
    # 1. Build the prompt
    prompt = f"""
You are an ESG news analysis assistant.

Use the article below to answer the question.

Article:
\"\"\"
{metadata[top_idx]["content"]}
\"\"\"

Question: {query}
Answer:
"""

    # 2. Tokenize and generate
    inputs  = flan_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    outputs = flan_model.generate(**inputs, max_new_tokens=max_new_tokens)
    # 3. Decode
    return flan_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [26]:
import torch

# 🔍 Check if CUDA is available
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("⚠️ CUDA not available. Running on CPU.")

# Set device safely
device = "cpu"

# ✅ Load FLAN-T5 and move to device
flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(device)

# Move SentenceTransformer to CPU
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

def generate_answer(query, top_k=1):
    # Step 1: Embed the query
    query_embedding = embedding_model.encode([query])

    # Step 2: Search FAISS index
    D, I = faiss_index.search(np.array(query_embedding), top_k)
    top_idx = I[0][0]

    # ✅ SAFETY CHECK
    if top_idx < 0:
        print("⚠️ No relevant article found in FAISS index.")
        return "No relevant article found to answer the query."

    # Step 3: Retrieve article from metadata
    retrieved_article = metadata[top_idx]["content"]

    # ✅ Truncate article to 400 words (to avoid long input issues)
    retrieved_article = " ".join(retrieved_article.split()[:400])

    # Step 4: Format prompt for FLAN-T5
    prompt = f"""
You are an ESG news analysis assistant.

Use the article below to answer the user's question based only on the content.

Article:
\"\"\"
{retrieved_article}
\"\"\"

Question: {query}
Answer:
"""

    # Step 5: Tokenize and generate response
    input_ids = flan_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).input_ids.to(device)
    output_ids = flan_model.generate(
        input_ids,
        max_length=256,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )
    answer = flan_tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Step 6: Fallback for unclear or short answers
    if len(answer.strip().split()) < 3:
        answer = "The article discusses ESG topics related to the company, but specific measures are not clearly mentioned."

    # Step 7: Print everything
    print("🔍 Retrieved Article:\n", retrieved_article)
    print("\n🧠 Answer:", answer)



# Example usage
query = "How is Exxon using technology to improve oil extraction?"
generate_answer(query)

CUDA Available: True
GPU Name: Tesla T4
🔍 Retrieved Article:
 Company: unknown Date: 2022-08-16 Headline: Tech Helps Big Oil Pump More, Belying Climate Pledges Text: It’s been a blockbuster summer for Big Oil. Exxon Mobil Corp. and Chevron Corp. posted record profits thanks to surging energy prices. The new US climate bill includes concessions to oil and gas companies. There are other, quieter beneficiaries: Microsoft Corp., Amazon.com Inc. and the other cloud-services companies that are increasingly responsible for the computing horsepower behind the oil giants’ efforts to find and extract more oil and natural gas. Among other things, Microsoft is making it possible for Exxon to analyze reams of oil field data. Amazon is helping drillers run simulations to maximize how much oil they can pump from existing wellsIt’s an awkward look for companies that have pledged to cut their own emissions. Microsoft has vowed to remove more carbon from the atmosphere than it emits by 2030, while Amazo